# Troubleshooting: Admin Login & Radio Public URL

Notebook ini berisi langkah-langkah pemeriksaan dan perintah untuk memperbaiki masalah login admin (`/admin`) dan konektivitas `RADIO_PUBLIC_URL`.

Ikuti langkah-langkah di setiap seksi dan jalankan perintah dari terminal yang sesuai (PowerShell pada Windows, shell pada Linux).

## 1. Periksa proses Node/PM2

Tujuan: Pastikan aplikasi LuxxBot berjalan di PM2 dan tidak crash.

Perintah (PowerShell/Unix):

```bash
# Tampilkan daftar proses PM2
pm2 status

# Detail satu aplikasi (ganti <app-name> dengan nama, misal 'luxx')
pm2 show <app-name>

# Lihat log terakhir (200 baris)
pm2 logs <app-name> --lines 200
```

## 2. Periksa port/listener (3920)

Tujuan: Pastikan proses mendengarkan port yang benar.

Perintah (Linux):

```bash
ss -tulpen | grep 3920
lsof -i :3920
netstat -plant | grep 3920
```

Perintah (Windows PowerShell):

```powershell
# Tampilkan proses yang mendengarkan port 3920
Get-NetTCPConnection -LocalPort 3920 -State Listen | Format-Table -AutoSize

# Alternatif: gunakan netstat
netstat -ano | findstr 3920
# lalu temukan PID dan lihat proses
Get-Process -Id <PID>
```

## 3. Tes endpoint lokal `/admin`

Tujuan: Periksa apakah endpoint admin dapat diakses lokal.

Perintah:

```bash
# Tes GET ke halaman admin (lihat status dan body)
curl -v http://localhost:3920/admin

# Tes menggunakan loopback explicit
curl -I http://127.0.0.1:3920/admin
```

Periksa header respons terutama `Access-Control-Allow-Origin` dan body error (Unauthorized / Admin API disabled).

## 4. Verifikasi `RADIO_PUBLIC_URL` vs `ADMIN_LINK` di `.env`

Tujuan: Pastikan `ADMIN_LINK` dan `RADIO_PUBLIC_URL` konsisten.

Perintah (jangan tampilkan nilai sensitif):

```bash
# Tampilkan hanya baris yang relevan dengan nomor baris
nl -ba .env | sed -n '1,200p' | grep -nE 'RADIO_PUBLIC_URL|ADMIN_LINK|ADMIN_API_TOKEN|ADMIN_CORS_ORIGIN'

# Atau cara portable: grep -nE 'RADIO_PUBLIC_URL|ADMIN_LINK' .env
```

Panduan:
- `RADIO_PUBLIC_URL` harus menunjuk ke URL publik server bot (contoh `https://xxx.trycloudflare.com`).
- `ADMIN_LINK` harus berupa `${RADIO_PUBLIC_URL}/admin` jika admin di-serve dari server bot.
- Jangan atur `ADMIN_LINK` ke GitHub Pages. Admin UI di GitHub Pages harus menggunakan `API Base` yang menunjuk ke `RADIO_PUBLIC_URL` (server bot).

## 5. Tes akses publik `RADIO_PUBLIC_URL`

Tujuan: Pastikan URL publik (tunnel) mengarah ke aplikasi dan dapat diakses.

Perintah:

```bash
# Tes root publik
curl -v "${RADIO_PUBLIC_URL}"

# Tes admin publik
curl -I "${RADIO_PUBLIC_URL}/admin"

# Cek traceroute (opsional)
traceroute $(echo ${RADIO_PUBLIC_URL} | sed -E 's_http(s)?://__')
```

Catatan: Jalankan dari mesin lain atau gunakan `curl --connect-timeout 5` dari remote host untuk memastikan routing publik.

## 6. Periksa tunnel / layanan VPS (cloudflared/ngrok)

Tujuan: Jika menggunakan tunnel, pastikan tunnel aktif dan meneruskan ke port 3920.

Perintah (cloudflared):

```bash
# Lihat status systemd (Linux)
sudo systemctl status cloudflared

# List tunnels (jika terpasang)
cloudflared tunnel list
```

Perintah (ngrok):

```bash
# Jalankan ngrok lokal untuk testing
ngrok http 3920
# atau cek status web UI http://127.0.0.1:4040/status
```

Jika tunnel tidak aktif, jalankan ulang service atau regenerate tunnel, lalu update `RADIO_PUBLIC_URL` di `.env`.

## 7. Periksa firewall / security group / iptables

Perintah:

```bash
# UFW (Ubuntu)
sudo ufw status verbose

# Iptables
sudo iptables -L -n -v

# Windows Firewall (PowerShell)
Get-NetFirewallRule | Where-Object { $_.Direction -eq 'Inbound' -and $_.Enabled -eq 'True' } | Format-Table -AutoSize
```

Pastikan port 3920 boleh diakses dari sumber yang diinginkan (loopback, LAN, publik sesuai kebutuhan).

## 8. Periksa reverse proxy (nginx/traefik)

Periksa konfigurasi dan pastikan proxy meneruskan ke `http://127.0.0.1:3920`.

Perintah (Linux):

```bash
# Cari referensi proxy_pass yang mengarah ke 3920
sudo grep -R "3920\|proxy_pass" /etc/nginx -n || true

# Test nginx config
sudo nginx -t

# Restart nginx
sudo systemctl restart nginx
```

Jika ada reverse proxy, pastikan header CORS & X-Forwarded-* di-passthrough dengan benar.

## 9. Periksa log aplikasi dan PM2

Kumpulkan log error dan cari pesan terkait CORS, env missing, atau crash.

Perintah:

```bash
pm2 logs <app-name> --lines 500
# atau khusus file log yang disimpan
# journalctl -u <service> --no-pager -n 200
```

Cari baris yang menyebut `Admin API`, `Admin API disabled`, `Failed to bind`, `EADDRINUSE`, atau error runtime lain.

In [ ]:
## 10. Script Python otomatis untuk cek endpoint & CORS

Skrip singkat untuk memeriksa status dan header CORS.

```python
import requests

urls = [
    'http://localhost:3920/admin',
    'http://127.0.0.1:3920/admin'
]
for u in urls:
    try:
        r = requests.get(u, timeout=5)
        print(u, r.status_code, r.headers.get('access-control-allow-origin'))
    except Exception as e:
        print(u, 'ERROR', e)
```

Jalankan dari mesin lokal dan, jika memungkinkan, dari remote host untuk membandingkan hasil.

## 11. Update `.env` dan restart service

Langkah singkat:

1. Edit `.env` jika `RADIO_PUBLIC_URL` atau `ADMIN_LINK` salah. Contoh yang benar:
   - `RADIO_PUBLIC_URL=https://your-tunnel.example.com`
   - `ADMIN_LINK=https://your-tunnel.example.com/admin`

2. Restart PM2 agar environment baru terbaca:

```bash
pm2 restart luxx --update-env
# atau jika nama app berbeda:
pm2 restart <app-name> --update-env
```

3. Verifikasi ulang dengan `curl` dan login dari browser.

Catatan: Jangan commit `.env` ke git. Selalu simpan backup sebelum ubah.
